# Reestructura de Campus Lab: pipeline de documentación

Este notebook documenta y reproduce todo el análisis que hemos hecho sobre la biblioteca de Documentos de SharePoint (`ProyectoBLP`):

0. Actualizar el extracto de SharePoint (`clDDMMYY.xlsx`) a partir de `query.iqy` — **solo Windows, con Excel instalado**
1. Construir el árbol completo de carpetas y archivos
2. Mapear el árbol actual contra la taxonomía propuesta (basada en el tablero de Miro)
3. Detectar candidatos a duplicado a nivel de archivo (nombre + tamaño)
4. Detectar candidatos a duplicado a nivel de carpeta (contenido idéntico)
5. Generar la infografía de resumen

Cada corrida nueva solo necesita un `clDDMMYY.xlsx` fresco en `entrada/`. El resto se regenera solo.

## 0. Actualizar el extracto desde SharePoint (solo Windows)

Esta celda abre `query.iqy` con Excel instalado en tu máquina, refresca la conexión con tu sesión ya autenticada, y guarda el resultado como `cl{DDMMYY}.xlsx` en `entrada/`.

**Requisitos:** Windows, Excel de escritorio instalado, `pip install pywin32`, y que `query.iqy` esté en la misma carpeta que este notebook (o ajusta `CARPETA_QUERY`).

**Qué puede pasar la primera vez:** si tu sesión de Excel no tiene credenciales frescas, te va a pedir iniciar sesión igual que cuando lo abres a mano. Deja `excel.Visible = True` para poder resolverlo tú mismo si aparece.

Esta celda no corre en Linux/Mac, se salta sola si el sistema operativo no es Windows.

In [ ]:
import os, platform
from datetime import date

CARPETA_QUERY = "entrada"        # donde vive query.iqy
CARPETA_DATOS_ENTRADA = "entrada"    # aquí también cae el cl{DDMMYY}.xlsx generado

def actualizar_desde_sharepoint(carpeta_query=CARPETA_QUERY, carpeta_destino=CARPETA_DATOS_ENTRADA):
    """Abre query.iqy en Excel, refresca la conexión y guarda cl{DDMMYY}.xlsx."""
    import win32com.client as win32

    os.makedirs(carpeta_destino, exist_ok=True)
    ruta_iqy = os.path.abspath(os.path.join(carpeta_query, "query.iqy"))
    nombre_salida = f"cl{date.today().strftime('%d%m%y')}.xlsx"
    ruta_salida = os.path.abspath(os.path.join(carpeta_destino, nombre_salida))

    if not os.path.exists(ruta_iqy):
        raise FileNotFoundError(f"No se encontró query.iqy en: {ruta_iqy}")

    excel = win32.gencache.EnsureDispatch("Excel.Application")
    excel.Visible = True       # por si pide iniciar sesión
    excel.DisplayAlerts = False
    try:
        wb = excel.Workbooks.Open(ruta_iqy)
        wb.RefreshAll()
        excel.CalculateUntilAsyncQueriesDone()
        wb.SaveAs(ruta_salida, FileFormat=51)  # 51 = xlOpenXMLWorkbook (.xlsx)
        wb.Close(SaveChanges=False)
        print(f"Guardado: {ruta_salida}")
        return ruta_salida
    finally:
        excel.Quit()

if platform.system() == "Windows":
    actualizar_desde_sharepoint()
else:
    print("Esta celda solo corre en Windows con Excel instalado. Se omite en este sistema.") # por qué lo haría para otro SO??? Soy becarie smh
    print("Coloca manualmente tu cl{DDMMYY}.xlsx más reciente en entrada/ y continúa.")

## 1. Cargar el extracto más reciente

Busca automáticamente el `cl{DDMMYY}.xlsx` más nuevo en `entrada/`, para que no tengas que cambiar el nombre del archivo a mano en cada corrida.

In [ ]:
import pandas as pd
import re
import glob
from datetime import datetime

def encontrar_extracto_mas_reciente(carpeta=CARPETA_DATOS_ENTRADA):
    candidatos = glob.glob(os.path.join(carpeta, "cl??????.xlsx"))
    con_fecha = []
    for ruta in candidatos:
        m = re.search(r"cl(\d{2})(\d{2})(\d{2})\.xlsx$", os.path.basename(ruta))
        if m:
            dd, mm, yy = m.groups()
            try:
                fecha = datetime.strptime(f"{dd}{mm}{yy}", "%d%m%y")
                con_fecha.append((fecha, ruta))
            except ValueError:
                continue
    if not con_fecha:
        raise FileNotFoundError(f"No se encontró ningún cl{{DDMMYY}}.xlsx en {carpeta}")
    con_fecha.sort()
    return con_fecha[-1]

fecha_extracto, RUTA_EXTRACTO = encontrar_extracto_mas_reciente()
print(f"Usando extracto del {fecha_extracto:%d/%m/%Y}: {RUTA_EXTRACTO}")

RAIZ_SITIO = "sites/ProyectoBLP/Documentos compartidos"
CARPETA_SALIDAS = "salida"
os.makedirs(CARPETA_SALIDAS, exist_ok=True)

df = pd.read_excel(RUTA_EXTRACTO)
df["ruta_rel"] = df["Ruta de acceso"].str.replace(RAIZ_SITIO, "", regex=False).str.strip("/")
df["ruta_completa"] = df.apply(lambda r: (r["ruta_rel"] + "/" + r["Nombre"]) if r["ruta_rel"] else r["Nombre"], axis=1)
print(f"{len(df):,} elementos cargados ({(df['Tipo de elemento']=='Elemento').sum():,} archivos, {(df['Tipo de elemento']=='Carpeta').sum():,} carpetas)")

## 2. Árbol completo

Separa la ruta en niveles, calcula tamaño legible y marca señales de revisión (nombres con error de codificación, carpetas provisionales, etc.).

In [ ]:
def fmt_size(b):
    if pd.isna(b):
        return ""
    b = float(b)
    for unit in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f} {unit}"
        b /= 1024
    return f"{b:.1f} TB"

PATRONES_BASURA = {
    "Posible error de codificación": r"[¦╠üó]",
    "Carpeta de sistema/temporal": r"__MACOSX|\.DS_Store|Thumbs\.db",
    "Carpeta provisional/duda": r"(?i)provisional|ignorar|nueva carpeta|duda nombre|prueba$",
    "Copia o revisión pendiente": r"(?i)^copias?$|^revisadas?$|reedici[oó]n|correcciones|^v\d+$|\(1\)|\(2\)|\(3\)",
    "Marcado como eliminado": r"(?i)eliminad",
}

def clasificar_senal(nombre):
    hits = [k for k, pat in PATRONES_BASURA.items() if re.search(pat, str(nombre))]
    return "; ".join(hits) if hits else ""

max_niveles = df["ruta_rel"].apply(lambda x: len(x.split("/")) if x else 0).max()
niveles = df["ruta_rel"].apply(lambda x: x.split("/") if x else [])
for i in range(max_niveles):
    df[f"Nivel{i+1}"] = niveles.apply(lambda l: l[i] if i < len(l) else "")

df["Tamaño legible"] = df["Tamaño de archivo"].apply(fmt_size)
df["Señal de revisión"] = df["Nombre"].apply(clasificar_senal)

nivel_cols = [f"Nivel{i+1}" for i in range(max_niveles)]
print(f"Profundidad máxima: {max_niveles} niveles")
print(f"Elementos con señal de revisión: {(df['Señal de revisión'] != '').sum()}")

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

HEADER_FONT = Font(name="Arial", bold=True, color="FFFFFF")
HEADER_FILL = PatternFill(start_color="2F5496", end_color="2F5496", fill_type="solid")
BODY_FONT = Font(name="Arial", size=10)
FLAG_FILL = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")

def escribir_hoja(ws, columnas, filas, resaltar_col=None, anchos=None):
    for c, h in enumerate(columnas, start=1):
        cell = ws.cell(row=1, column=c, value=h)
        cell.font, cell.fill = HEADER_FONT, HEADER_FILL
    for r, row in enumerate(filas, start=2):
        for c, val in enumerate(row, start=1):
            ws.cell(row=r, column=c, value=str(val) if pd.notna(val) else "").font = BODY_FONT
        if resaltar_col is not None and row[resaltar_col]:
            for c in range(1, len(columnas) + 1):
                ws.cell(row=r, column=c).fill = FLAG_FILL
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    if anchos:
        for i, w in enumerate(anchos, start=1):
            ws.column_dimensions[get_column_letter(i)].width = w

cols_arbol = nivel_cols + ["Nombre", "Tipo de elemento", "Tamaño legible", "Creado", "Modificado", "Modificado por", "Señal de revisión", "ruta_completa"]
df_arbol = df[cols_arbol].sort_values(nivel_cols + ["Nombre"]).reset_index(drop=True)

wb = Workbook()
ws1 = wb.active
ws1.title = "Árbol completo"
escribir_hoja(ws1, cols_arbol, df_arbol.itertuples(index=False), resaltar_col=-2,
              anchos=[16]*max_niveles + [40, 12, 14, 18, 18, 24, 30, 60])

ws2 = wb.create_sheet("Resumen nivel 1")
resumen1 = df.groupby("Nivel1").agg(elementos=("Nombre", "size"), peso=("Tamaño de archivo", "sum")).sort_values("elementos", ascending=False).reset_index()
resumen1.loc[resumen1["Nivel1"] == "", "Nivel1"] = "(raíz)"
total = len(df)
filas2 = [[r.Nivel1, int(r.elementos), f"{r.elementos/total*100:.1f}%", fmt_size(r.peso)] for r in resumen1.itertuples(index=False)]
escribir_hoja(ws2, ["Carpeta raíz", "Cantidad de elementos", "% del total", "Peso total"], filas2, anchos=[40, 20, 12, 14])

ws3 = wb.create_sheet("Señales de revisión")
df_flags = df[df["Señal de revisión"] != ""][nivel_cols + ["Nombre", "Tipo de elemento", "Tamaño legible", "Señal de revisión", "ruta_completa"]]
escribir_hoja(ws3, list(df_flags.columns), df_flags.itertuples(index=False), anchos=[16]*max_niveles + [40, 12, 14, 34, 60])

ruta_arbol = os.path.join(CARPETA_SALIDAS, "Arbol_SharePoint_ProyectoBLP.xlsx")
wb.save(ruta_arbol)
print(f"Guardado: {ruta_arbol}")

## 3. Propuesta de taxonomía

El diccionario `MAPEO` es la decisión editorial: qué carpeta actual corresponde a qué línea de trabajo del Miro. Si Maya pide ajustar una categoría, este es el único lugar que hay que tocar; el resto del notebook se recalcula solo.

In [ ]:
MAPEO = {
    ("(raíz)", "(sin subcarpeta)"): ("15. Sin clasificar / por revisar", "Archivos sueltos en la raíz de Documentos, sin carpeta."),
    ("Actividades e intervenciones", "Concurso Naturalista"): ("08. Actividades y Eventos de Difusión", "CONSOLIDAR con CAMPUS LAB (Maya)/Concurso Naturalista y Naturalista ITESO (confirmado por Remi: misma iniciativa)."),
    ("Actividades e intervenciones", "Guías y mapas"): ("05. Producción de Información", ""),
    ("Actividades e intervenciones", "Avistamientos y recorridos"): ("06. Avistamientos y Recorridos", "CONSOLIDAR con CAMPUS LAB (Maya)/2. Avistamientos."),
    ("Actividades e intervenciones", "Naturalista ITESO"): ("08. Actividades y Eventos de Difusión", "CONFIRMADO: mismo proyecto que Concurso Naturalista."),
    ("Actividades e intervenciones", "Info espacios"): ("05. Producción de Información", "Fichas informativas de espacios del campus."),
    ("Actividades e intervenciones", "(sin subcarpeta)"): ("15. Sin clasificar / por revisar", ""),
    ("Actividades e intervenciones", "Intervenciones universitarias"): ("08. Actividades y Eventos de Difusión", ""),
    ("Actividades e intervenciones", "Actividades del CPC para CL"): ("08. Actividades y Eventos de Difusión", "CPC = Centro de Promoción Cultural."),
    ("Actividades e intervenciones", "Levantamiento de información para Campus Lab"): ("01. Gobernanza y Planeación", ""),
    ("Actividades e intervenciones", "Jardín meditativo"): ("08. Actividades y Eventos de Difusión", ""),
    ("Aplicaciones", "Microsoft Forms"): ("14. Documentos Institucionales de Referencia", "Formularios institucionales."),
    ("Aplicaciones", "(sin subcarpeta)"): ("15. Sin clasificar / por revisar", ""),
    ("Bases de datos", "FOTOS"): ("11. Fototeca (staging)", "Espejo temporal del volumen de fotos real."),
    ("Bases de datos", "SIG MAPA"): ("10. Identidad Gráfica y Cartografía", "CONSOLIDAR con SIG/SIG MAPA."),
    ("Bases de datos", "audios AVES"): ("05. Producción de Información", ""),
    ("Bases de datos", "Audio y fotos recorrido por el campus, Rodolfo con rector y equipo de rectoría"): ("08. Actividades y Eventos de Difusión", ""),
    ("Bases de datos", "(sin subcarpeta)"): ("15. Sin clasificar / por revisar", ""),
    ("Bases de datos", "Especies Casa Clavigero"): ("05. Producción de Información", ""),
    ("Bases de datos", "EXCEL"): ("12. Bases de Datos Tabulares", ""),
    ("Bases de datos", "asistencia eventos (listas)"): ("08. Actividades y Eventos de Difusión", ""),
    ("Becarios", "Becario (SC) Santiago ornelas"): ("13. Becarios y Equipo", ""),
    ("Becarios", "AVES_TrípticoObservación"): ("05. Producción de Información", "MAL UBICADA dentro de Becarios."),
    ("Becarios", "(sin subcarpeta)"): ("13. Becarios y Equipo", ""),
    ("CAMPUS LAB (Maya)", "3. Proyecto web aves iteso"): ("04. Sitio Web Campus Lab", "CONSOLIDAR con Página web Campus Lab/Sitio Aves."),
    ("CAMPUS LAB (Maya)", "Concurso Naturalista"): ("08. Actividades y Eventos de Difusión", "CONSOLIDAR (ver nota arriba)."),
    ("CAMPUS LAB (Maya)", "5. Proyecto libro Aves ITESO"): ("05. Producción de Información", ""),
    ("CAMPUS LAB (Maya)", "2. Avistamientos"): ("06. Avistamientos y Recorridos", "CONSOLIDAR (ver nota arriba)."),
    ("CAMPUS LAB (Maya)", "1. Campus lab colaborativo"): ("01. Gobernanza y Planeación", "Revisar contenido."),
    ("CAMPUS LAB (Maya)", "Yincana"): ("08. Actividades y Eventos de Difusión", ""),
    ("CAMPUS LAB (Maya)", "Actividades diversas"): ("08. Actividades y Eventos de Difusión", ""),
    ("CAMPUS LAB (Maya)", "4. Actividades sobre aves"): ("08. Actividades y Eventos de Difusión", ""),
    ("CAMPUS LAB (Maya)", "(sin subcarpeta)"): ("15. Sin clasificar / por revisar", ""),
    ("CAMPUS LAB (Maya)", "Derechos de autor sobre recursos"): ("11. Fototeca (staging)", "Vincular como registro maestro de autoría."),
    ("CAMPUS LAB (Maya)", "Documentos de referencia"): ("14. Documentos Institucionales de Referencia", "CONSOLIDAR: 3 ubicaciones distintas."),
    ("CAMPUS LAB (Maya)", "Documentos de gestión"): ("01. Gobernanza y Planeación", ""),
    ("Documentos base del programa CL", "(sin subcarpeta)"): ("01. Gobernanza y Planeación", ""),
    ("Documentos de referencia", "(sin subcarpeta)"): ("14. Documentos Institucionales de Referencia", "CONSOLIDAR (ver nota arriba)."),
    ("Documentos de referencia", "Sobre comunicación de la ciencia"): ("14. Documentos Institucionales de Referencia", ""),
    ("Documentos de referencia", "Documentos institucionales de referencia"): ("14. Documentos Institucionales de Referencia", ""),
    ("Documentos sobre aves", "(sin subcarpeta)"): ("05. Producción de Información", ""),
    ("General", "(sin subcarpeta)"): ("15. Sin clasificar / por revisar", ""),
    ("Identidad gráfica Campus lab", "Publicidad"): ("10. Identidad Gráfica y Cartografía", ""),
    ("Identidad gráfica Campus lab", "(sin subcarpeta)"): ("10. Identidad Gráfica y Cartografía", ""),
    ("Identidad gráfica Campus lab", "Naturalista"): ("08. Actividades y Eventos de Difusión", "Material gráfico del Concurso/Naturalista ITESO."),
    ("Identidad gráfica Campus lab", "Primera iniciativa"): ("15. Sin clasificar / por revisar", "Posible material obsoleto, confirmar vigencia."),
    ("Proyecto BLP", "(sin subcarpeta)"): ("01. Gobernanza y Planeación", ""),
    ("Proyecto Museológico y Museográfico", "(sin subcarpeta)"): ("02. Proyecto Museográfico", ""),
    ("Página web Campus Lab", "Sitio Aves"): ("04. Sitio Web Campus Lab", "CONSOLIDAR (ver nota arriba)."),
    ("Página web Campus Lab", "(sin subcarpeta)"): ("04. Sitio Web Campus Lab", ""),
    ("Página web Campus Lab", "Arboretums"): ("04. Sitio Web Campus Lab", ""),
    ("Página web Campus Lab", "Ciclo hidrológico en el iteso"): ("04. Sitio Web Campus Lab", "Ámbito temático: Agua."),
    ("SIG", "SIG MAPA"): ("10. Identidad Gráfica y Cartografía", "CONSOLIDAR (ver nota arriba)."),
    ("SIG", "(sin subcarpeta)"): ("10. Identidad Gráfica y Cartografía", ""),
    ("Textos y publicaciones sobre el proyecto CL", "(sin subcarpeta)"): ("05. Producción de Información", ""),
}

df["nivel1_map"] = df["ruta_rel"].apply(lambda x: (x.split("/") if x else [""])[0] or "(raíz)")
df["nivel2_map"] = df["ruta_rel"].apply(lambda x: (x.split("/") + [""])[1] if x else "").replace("", "(sin subcarpeta)")

filas_mapeo = []
for (n1, n2), grp in df.groupby(["nivel1_map", "nivel2_map"]):
    n, peso = len(grp), grp["Tamaño de archivo"].sum()
    cat, nota = MAPEO.get((n1, n2), ("15. Sin clasificar / por revisar", "SIN MAPEO MANUAL: revisar caso por caso."))
    filas_mapeo.append([n1, n2, n, peso, cat, nota])

mapeo_df = pd.DataFrame(filas_mapeo, columns=["Carpeta raíz actual", "Subcarpeta actual", "Cantidad de elementos", "peso_bytes", "Categoría propuesta", "Nota"])
mapeo_df["Peso legible"] = mapeo_df["peso_bytes"].apply(fmt_size)
mapeo_df = mapeo_df.sort_values(["Categoría propuesta", "Cantidad de elementos"], ascending=[True, False]).reset_index(drop=True)

resumen_cat = mapeo_df.groupby("Categoría propuesta").agg(elementos=("Cantidad de elementos", "sum"), peso=("peso_bytes", "sum")).sort_values("elementos", ascending=False).reset_index()
resumen_cat["% del total"] = (resumen_cat["elementos"] / resumen_cat["elementos"].sum() * 100).round(1)
resumen_cat["Peso legible"] = resumen_cat["peso"].apply(fmt_size)
print(resumen_cat[["Categoría propuesta", "elementos", "% del total", "Peso legible"]].to_string(index=False))

In [ ]:
wb = Workbook()
ws1 = wb.active
ws1.title = "Taxonomía propuesta"
filas_r = [[row["Categoría propuesta"], int(row["elementos"]), f"{row['% del total']}%", row["Peso legible"]] for _, row in resumen_cat.iterrows()]
escribir_hoja(ws1, ["Categoría propuesta", "Cantidad de elementos", "% del total", "Peso total"], filas_r, anchos=[42, 20, 12, 14])

ws2 = wb.create_sheet("Mapeo detallado")
cols_map = ["Carpeta raíz actual", "Subcarpeta actual", "Cantidad de elementos", "Peso legible", "Categoría propuesta", "Nota"]
CONSOLIDAR_FILL = PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid")
for c, h in enumerate(cols_map, start=1):
    cell = ws2.cell(row=1, column=c, value=h); cell.font, cell.fill = HEADER_FONT, HEADER_FILL
for r, row in enumerate(mapeo_df[cols_map].itertuples(index=False), start=2):
    for c, val in enumerate(row, start=1):
        ws2.cell(row=r, column=c, value=val).font = BODY_FONT
    if "CONSOLIDAR" in str(row[-1]) or "CONFIRMADO" in str(row[-1]):
        for c in range(1, len(cols_map) + 1):
            ws2.cell(row=r, column=c).fill = CONSOLIDAR_FILL
ws2.freeze_panes, ws2.auto_filter.ref = "A2", ws2.dimensions
for i, w in enumerate([32, 32, 12, 13, 34, 75], start=1):
    ws2.column_dimensions[get_column_letter(i)].width = w

ruta_taxonomia = os.path.join(CARPETA_SALIDAS, "Propuesta_Taxonomia_CampusLab.xlsx")
wb.save(ruta_taxonomia)
print(f"Guardado: {ruta_taxonomia}")

## 4. Candidatos a duplicado por archivo

Fotos con el mismo nombre en más de una carpeta, separadas por nivel de confianza según coincida también el tamaño exacto.

In [ ]:
EXT_IMAGEN = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".nef", ".cr2", ".cr3", ".raw", ".heic", ".bmp", ".gif", ".psd"}

archivos = df[df["Tipo de elemento"] == "Elemento"].copy()
archivos["ext"] = archivos["Nombre"].apply(lambda x: os.path.splitext(str(x))[1].lower())
fotos = archivos[archivos["ext"].isin(EXT_IMAGEN)].copy()

conteo = fotos.groupby("Nombre")["ruta_rel"].nunique()
nombres_rep = conteo[conteo > 1].index
dup = fotos[fotos["Nombre"].isin(nombres_rep)].copy()

tam_distintos = dup.groupby("Nombre")["Tamaño de archivo"].nunique()
dup["confianza"] = dup["Nombre"].map(tam_distintos).apply(
    lambda n: "Alta: mismo nombre y mismo tamaño en todas sus copias" if n == 1 else "Revisar: mismo nombre, tamaño distinto entre copias"
)
dup["Tamaño legible"] = dup["Tamaño de archivo"].apply(fmt_size)
dup["n_apariciones"] = dup["Nombre"].map(dup.groupby("Nombre")["ruta_rel"].nunique())

n_alta = (tam_distintos == 1).sum()
n_revisar = (tam_distintos > 1).sum()
arch_alta = dup[dup["confianza"].str.startswith("Alta")].shape[0]
arch_revisar = dup[dup["confianza"].str.startswith("Revisar")].shape[0]
peso_alta_dup = dup[dup["confianza"].str.startswith("Alta")].groupby("Nombre")["Tamaño de archivo"].agg(lambda s: s.iloc[0] * (len(s) - 1)).sum()

print(f"Alta confianza: {n_alta} nombres, {arch_alta} archivos")
print(f"Revisar: {n_revisar} nombres, {arch_revisar} archivos")
print(f"Espacio recuperable estimado (alta confianza): {fmt_size(peso_alta_dup)}")

In [ ]:
detalle_dup = dup[["Nombre", "n_apariciones", "confianza", "ruta_rel", "Tamaño legible", "Modificado", "Modificado por"]].copy()
detalle_dup = detalle_dup.rename(columns={"ruta_rel": "Carpeta contenedora"})
detalle_dup = detalle_dup.sort_values(["confianza", "n_apariciones", "Nombre", "Carpeta contenedora"], ascending=[True, False, True, True])

wb = Workbook()
ws1 = wb.active
ws1.title = "Resumen"
ws1["A1"] = "Candidatos a duplicado: fotos con el mismo nombre en más de una carpeta"
ws1["A1"].font = Font(name="Arial", bold=True, size=12)
textos = [
    (3, f"Nombres con ALTA confianza (mismo nombre y mismo tamaño exacto en todas sus copias): {n_alta}", True),
    (4, f"  → archivos involucrados: {arch_alta}", False),
    (5, f"  → espacio recuperable estimado: {fmt_size(peso_alta_dup)}", False),
    (7, f"Nombres para REVISAR (mismo nombre, tamaño distinto): {n_revisar}", True),
    (8, f"  → archivos involucrados: {arch_revisar}", False),
]
for row, txt, bold in textos:
    c = ws1.cell(row=row, column=1, value=txt)
    c.font = Font(name="Arial", size=11, bold=bold)
ws1["A10"] = "Nota: 'Alta confianza' es altísima probabilidad estadística, no confirmación absoluta (eso requiere comparar contenido byte a byte)."
ws1["A10"].font = Font(name="Arial", italic=True, size=10, color="808080")
ws1.column_dimensions["A"].width = 100

ws2 = wb.create_sheet("Detalle por ubicación")
ALTA_FILL = PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid")
cols_d = list(detalle_dup.columns)
for c, h in enumerate(cols_d, start=1):
    cell = ws2.cell(row=1, column=c, value=h); cell.font, cell.fill = HEADER_FONT, HEADER_FILL
for r, row in enumerate(detalle_dup.itertuples(index=False), start=2):
    for c, val in enumerate(row, start=1):
        ws2.cell(row=r, column=c, value=str(val) if pd.notna(val) else "").font = BODY_FONT
    if str(row[2]).startswith("Alta"):
        for c in range(1, len(cols_d) + 1):
            ws2.cell(row=r, column=c).fill = ALTA_FILL
ws2.freeze_panes, ws2.auto_filter.ref = "A2", ws2.dimensions
for i, w in enumerate([35, 12, 45, 60, 12, 18, 26], start=1):
    ws2.column_dimensions[get_column_letter(i)].width = w

ruta_dup_archivo = os.path.join(CARPETA_SALIDAS, "Candidatos_Duplicados_Fotos.xlsx")
wb.save(ruta_dup_archivo)
print(f"Guardado: {ruta_dup_archivo}")

## 5. Candidatos a duplicado por carpeta

Carpetas cuyo contenido directo (nombre + tamaño de cada archivo que vive adentro) es 100% idéntico al de otra carpeta, sin importar en qué nivel del árbol estén. Solo se consideran carpetas con 3 o más archivos directos, para evitar falsos positivos triviales.

In [ ]:
from collections import defaultdict

archivos_por_carpeta = archivos.groupby("ruta_rel").apply(lambda g: frozenset(zip(g["Nombre"], g["Tamaño de archivo"])))
carpetas_con_contenido = archivos_por_carpeta[archivos_por_carpeta.apply(len) >= 3]

firma_to_carpetas = defaultdict(list)
for carpeta, firma in carpetas_con_contenido.items():
    firma_to_carpetas[firma].append(carpeta)
grupos_dup = {f: c for f, c in firma_to_carpetas.items() if len(c) > 1}

filas_carpetas = []
total_carpetas_redundantes = total_archivos_redundantes = peso_redundante = 0
for i, (firma, carpetas) in enumerate(sorted(grupos_dup.items(), key=lambda kv: -len(kv[0]) * len(kv[1])), start=1):
    n_archivos, n_carpetas = len(firma), len(carpetas)
    peso_una_copia = sum(t for _, t in firma if pd.notna(t))
    sobrantes = n_carpetas - 1
    total_carpetas_redundantes += sobrantes
    total_archivos_redundantes += n_archivos * sobrantes
    peso_redundante += peso_una_copia * sobrantes
    for c in carpetas:
        filas_carpetas.append([i, n_archivos, n_carpetas, fmt_size(peso_una_copia), c])

print(f"Grupos de carpetas duplicadas: {len(grupos_dup)}")
print(f"Carpetas redundantes: {total_carpetas_redundantes}")
print(f"Archivos redundantes: {total_archivos_redundantes}")
print(f"Espacio recuperable estimado: {fmt_size(peso_redundante)}")

In [ ]:
detalle_carpetas_df = pd.DataFrame(filas_carpetas, columns=["Grupo", "Archivos por copia", "Copias encontradas", "Peso de una copia", "Ruta de la carpeta"])

wb = Workbook()
ws1 = wb.active
ws1.title = "Resumen"
ws1["A1"] = "Carpetas con contenido 100% idéntico"
ws1["A1"].font = Font(name="Arial", bold=True, size=12)
for row, txt in [
    (3, f"Grupos de carpetas duplicadas: {len(grupos_dup)}"),
    (4, f"Carpetas redundantes: {total_carpetas_redundantes}"),
    (5, f"Archivos redundantes: {total_archivos_redundantes}"),
    (6, f"Espacio recuperable estimado: {fmt_size(peso_redundante)}"),
]:
    ws1.cell(row=row, column=1, value=txt).font = Font(name="Arial", size=11, bold=True)
ws1["A8"] = "Nota: este número no se suma al de duplicados por archivo, se traslapan. Compara solo archivos directos de cada carpeta, sin bajar a subcarpetas anidadas."
ws1["A8"].font = Font(name="Arial", italic=True, size=10, color="808080")
ws1.column_dimensions["A"].width = 100

ws2 = wb.create_sheet("Detalle por grupo")
escribir_hoja(ws2, list(detalle_carpetas_df.columns), detalle_carpetas_df.itertuples(index=False), anchos=[10, 16, 16, 16, 100])

ruta_dup_carpeta = os.path.join(CARPETA_SALIDAS, "Carpetas_Duplicadas.xlsx")
wb.save(ruta_dup_carpeta)
print(f"Guardado: {ruta_dup_carpeta}")

## 6. Infografía de resumen

Versión estática (matplotlib) para documentación y reproducibilidad. La versión interactiva que se mostró en el chat vive aparte, esta es la que queda archivada junto con el análisis.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["font.family"] = "DejaVu Sans"

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# --- Antes: por carpeta raíz ---
antes = resumen1.set_index("Nivel1")["elementos"].sort_values()
top_antes = antes.tail(8)
otras_antes = antes.iloc[:-8].sum()
if otras_antes > 0:
    top_antes = pd.concat([pd.Series({f"Otras ({len(antes)-8} carpetas)": otras_antes}), top_antes])
axes[0].barh(top_antes.index, top_antes.values, color="#2a78d6")
axes[0].set_title("Cómo está hoy\n(elementos por carpeta raíz)", fontsize=12, fontweight="bold")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for i, v in enumerate(top_antes.values):
    axes[0].text(v, i, f" {v:,}", va="center", fontsize=9)

# --- Propuesta: por categoría de taxonomía ---
prop = resumen_cat.set_index("Categoría propuesta")["elementos"].sort_values()
axes[1].barh(prop.index, prop.values, color="#1baf7a")
axes[1].set_title("Propuesta de taxonomía\n(elementos por línea de trabajo)", fontsize=12, fontweight="bold")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for i, v in enumerate(prop.values):
    axes[1].text(v, i, f" {v:,}", va="center", fontsize=9)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
ruta_fig1 = os.path.join(CARPETA_SALIDAS, "infografia_antes_vs_propuesta.png")
plt.savefig(ruta_fig1, dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado: {ruta_fig1}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
etiquetas = ["Total\ninventariado", "Concentrado en\n2 carpetas", "Nombres de foto\nrepetidos", "Fotos\ninvolucradas"]
valores = [len(df), 82, len(nombres_rep), arch_alta + arch_revisar]
sufijos = ["", "%", "", f" ({(arch_alta+arch_revisar)/len(fotos)*100:.0f}%)"]

ax.axis("off")
for i, (et, val, suf) in enumerate(zip(etiquetas, valores, sufijos)):
    x = i / len(etiquetas) + 0.08
    ax.text(x, 0.55, f"{val:,}{suf}", fontsize=22, fontweight="bold", transform=ax.transAxes)
    ax.text(x, 0.30, et, fontsize=10, color="#52514e", transform=ax.transAxes)

plt.tight_layout()
ruta_fig2 = os.path.join(CARPETA_SALIDAS, "infografia_metricas_clave.png")
plt.savefig(ruta_fig2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado: {ruta_fig2}")

print()
print("=" * 60)
print("RESUMEN FINAL DE LA CORRIDA")
print("=" * 60)
print(f"Fecha del extracto usado: {fecha_extracto:%d/%m/%Y}")
print(f"Elementos inventariados: {len(df):,}")
print(f"Grupos de carpetas 100% duplicadas: {len(grupos_dup)}")
print(f"Nombres de foto repetidos (alta confianza): {n_alta}")
print()
print("Archivos generados en 'salida/':")
for f in sorted(os.listdir(CARPETA_SALIDAS)):
    print(f"  - {f}")